<a href="https://colab.research.google.com/github/shashithenuwara/IOT/blob/main/MLmodel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install firebase-admin pandas

In [2]:
from google.colab import files
uploaded = files.upload()

Saving iotdba-firebase-adminsdk-fbsvc-9b342554a8.json to iotdba-firebase-adminsdk-fbsvc-9b342554a8.json


In [24]:
import firebase_admin
from firebase_admin import credentials, db
import pandas as pd

# Path to your downloaded service account file
SERVICE_ACCOUNT_FILE = "iotdba-firebase-adminsdk-fbsvc-9b342554a8.json"

# Your Firebase Realtime Database URL
DATABASE_URL = "https://iotdba-default-rtdb.firebaseio.com/"

# Initialize only once
if not firebase_admin._apps:
    cred = credentials.Certificate(SERVICE_ACCOUNT_FILE)
    firebase_admin.initialize_app(cred, {
        "databaseURL": DATABASE_URL
    })

# Example: read from a node called 'sensor_data'
ref = db.reference("/logs")
data = ref.get()

print(type(data))
print(data)

<class 'dict'>
{'-Oqk0awQcHb_miuPnopu': {'air_quality_ppm': 176.5871, 'date': '2026-04-21', 'humidity': 75.6, 'light': 0.833333, 'status': 'CRITICAL', 'temperature': 29.4375, 'time': '18:01:43', 'timestamp': '2026-04-21 18:01:43'}, '-Oqk0eiMekEmU_mr2i2B': {'air_quality_ppm': 174.7966, 'date': '2026-04-21', 'humidity': 75.2, 'light': 0.833333, 'status': 'CRITICAL', 'temperature': 29.4375, 'time': '18:01:57', 'timestamp': '2026-04-21 18:01:57'}, '-Oqk1-ebRgI2Jcc2er0k': {'air_quality_ppm': 170.3727, 'date': '2026-04-21', 'humidity': 74.5, 'light': 0.833333, 'status': 'CRITICAL', 'temperature': 29.4375, 'time': '18:03:28', 'timestamp': '2026-04-21 18:03:28'}, '-Oqk132q5e3Wgsl0-306': {'air_quality_ppm': 172.1333, 'date': '2026-04-21', 'humidity': 74.4, 'light': 0.833333, 'status': 'CRITICAL', 'temperature': 29.375, 'time': '18:03:42', 'timestamp': '2026-04-21 18:03:42'}, '-Oqk16TBEYzhXoFLYdK2': {'air_quality_ppm': 159.2145, 'date': '2026-04-21', 'humidity': 74.4, 'light': 133.3333, 'status'

In [25]:
df = pd.DataFrame.from_dict(data, orient="index")
df.head(40)
df.shape

(7046, 8)

In [26]:
#Ensure columns are numeric
df["temperature"] = pd.to_numeric(df["temperature"], errors="coerce")
df["humidity"] = pd.to_numeric(df["humidity"], errors="coerce")
df["light"] = pd.to_numeric(df["light"], errors="coerce")
df["air_quality_ppm"] = pd.to_numeric(df["air_quality_ppm"], errors="coerce")

In [28]:
#Drop Null vals
df = df.dropna()
df.shape

(7046, 8)

In [33]:
#Naming the first column
df = pd.DataFrame.from_dict(data, orient="index").reset_index().rename(columns={"index": "record_id"})

In [34]:
#Feature Engineering - Add new column
df["temp_critical"] = (df["temperature"] >= 30).astype(int)
df["hum_critical"] = (df["humidity"] >= 65).astype(int)
df["light_critical"] = (df["light"] >= 300).astype(int)
df["air_critical"] = (df["air_quality_ppm"] >= 1000).astype(int)
def get_critical_feature(row):
    critical = []

    if row["temp_critical"] == 1:
        critical.append("temperature")
    if row["hum_critical"] == 1:
        critical.append("humidity")
    if row["light_critical"] == 1:
        critical.append("light")
    if row["air_critical"] == 1:
        critical.append("air_quality_ppm")

    return ",".join(critical) if critical else "normal"

df["critical_feature"] = df.apply(get_critical_feature, axis=1)
df.head(40)

,record_id,air_quality_ppm,date,humidity,light,status,temperature,time,timestamp,temp_critical,hum_critical,light_critical,air_critical,critical_feature
0,-Oqk0awQcHb_miuPnopu,176.5871,2026-04-21,75.6,0.833333,CRITICAL,29.4375,18:01:43,2026-04-21 18:01:43,0,1,0,0,humidity
1,-Oqk0eiMekEmU_mr2i2B,174.7966,2026-04-21,75.2,0.833333,CRITICAL,29.4375,18:01:57,2026-04-21 18:01:57,0,1,0,0,humidity
2,-Oqk1-ebRgI2Jcc2er0k,170.3727,2026-04-21,74.5,0.833333,CRITICAL,29.4375,18:03:28,2026-04-21 18:03:28,0,1,0,0,humidity
3,-Oqk132q5e3Wgsl0-306,172.1333,2026-04-21,74.4,0.833333,CRITICAL,29.3750,18:03:42,2026-04-21 18:03:42,0,1,0,0,humidity
4,-Oqk16TBEYzhXoFLYdK2,159.2145,2026-04-21,74.4,133.333300,CRITICAL,29.4375,18:03:56,2026-04-21 18:03:56,0,1,0,0,humidity
5,-Oqk19xT3H0_G3sNmugT,173.0182,2026-04-21,74.6,1499.167000,CRITICAL,29.4375,18:04:10,2026-04-21 18:04:10,0,1,1,0,"humidity,light"
6,-Oqk1DPIsyV0A-4pYogn,173.9059,2026-04-21,74.5,54.166660,CRITICAL,29.4375,18:04:24,2026-04-21 18:04:24,0,1,0,0,humidity
7,-Oqk1Gp6Cj-2_zZJse1L,168.6238,2026-04-21,74.5,37.500000,CRITICAL,30.3125,18:04:39,2026-04-21 18:04:39,1,1,0,0,"temperature,humidity"
8,-Oqk1KGW0aslWYBH_88k,173.0182,2026-04-21,74.5,37.500000,CRITICAL,32.9375,18:04:53,2026-04-21 18:04:53,1,1,0,0,"temperature,humidity"
9,-Oqk1NeaUaPz_f9OSjNd,173.0182,2026-04-21,74.5,36.666660,CRITICAL,33.8125,18:05:07,2026-04-21 18:05:07,1,1,0,0,"temperature,humidity"


,record_id,air_quality_ppm,date,humidity,light,status,temperature,time,timestamp
0,-Oqk0awQcHb_miuPnopu,176.5871,2026-04-21,75.6,0.833333,CRITICAL,29.4375,18:01:43,2026-04-21 18:01:43
1,-Oqk0eiMekEmU_mr2i2B,174.7966,2026-04-21,75.2,0.833333,CRITICAL,29.4375,18:01:57,2026-04-21 18:01:57
2,-Oqk1-ebRgI2Jcc2er0k,170.3727,2026-04-21,74.5,0.833333,CRITICAL,29.4375,18:03:28,2026-04-21 18:03:28
3,-Oqk132q5e3Wgsl0-306,172.1333,2026-04-21,74.4,0.833333,CRITICAL,29.3750,18:03:42,2026-04-21 18:03:42
4,-Oqk16TBEYzhXoFLYdK2,159.2145,2026-04-21,74.4,133.333300,CRITICAL,29.4375,18:03:56,2026-04-21 18:03:56
5,-Oqk19xT3H0_G3sNmugT,173.0182,2026-04-21,74.6,1499.167000,CRITICAL,29.4375,18:04:10,2026-04-21 18:04:10
6,-Oqk1DPIsyV0A-4pYogn,173.9059,2026-04-21,74.5,54.166660,CRITICAL,29.4375,18:04:24,2026-04-21 18:04:24
7,-Oqk1Gp6Cj-2_zZJse1L,168.6238,2026-04-21,74.5,37.500000,CRITICAL,30.3125,18:04:39,2026-04-21 18:04:39
8,-Oqk1KGW0aslWYBH_88k,173.0182,2026-04-21,74.5,37.500000,CRITICAL,32.9375,18:04:53,2026-04-21 18:04:53
9,-Oqk1NeaUaPz_f9OSjNd,173.0182,2026-04-21,74.5,36.666660,CRITICAL,33.8125,18:05:07,2026-04-21 18:05:07


In [37]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df["critical_feature_encoded"] = le.fit_transform(df["critical_feature"])
df.head(40)

,record_id,air_quality_ppm,date,humidity,light,status,temperature,time,timestamp,temp_critical,hum_critical,light_critical,air_critical,critical_feature,critical_feature_encoded
0,-Oqk0awQcHb_miuPnopu,176.5871,2026-04-21,75.6,0.833333,CRITICAL,29.4375,18:01:43,2026-04-21 18:01:43,0,1,0,0,humidity,0
1,-Oqk0eiMekEmU_mr2i2B,174.7966,2026-04-21,75.2,0.833333,CRITICAL,29.4375,18:01:57,2026-04-21 18:01:57,0,1,0,0,humidity,0
2,-Oqk1-ebRgI2Jcc2er0k,170.3727,2026-04-21,74.5,0.833333,CRITICAL,29.4375,18:03:28,2026-04-21 18:03:28,0,1,0,0,humidity,0
3,-Oqk132q5e3Wgsl0-306,172.1333,2026-04-21,74.4,0.833333,CRITICAL,29.3750,18:03:42,2026-04-21 18:03:42,0,1,0,0,humidity,0
4,-Oqk16TBEYzhXoFLYdK2,159.2145,2026-04-21,74.4,133.333300,CRITICAL,29.4375,18:03:56,2026-04-21 18:03:56,0,1,0,0,humidity,0
5,-Oqk19xT3H0_G3sNmugT,173.0182,2026-04-21,74.6,1499.167000,CRITICAL,29.4375,18:04:10,2026-04-21 18:04:10,0,1,1,0,"humidity,light",1
6,-Oqk1DPIsyV0A-4pYogn,173.9059,2026-04-21,74.5,54.166660,CRITICAL,29.4375,18:04:24,2026-04-21 18:04:24,0,1,0,0,humidity,0
7,-Oqk1Gp6Cj-2_zZJse1L,168.6238,2026-04-21,74.5,37.500000,CRITICAL,30.3125,18:04:39,2026-04-21 18:04:39,1,1,0,0,"temperature,humidity",4
8,-Oqk1KGW0aslWYBH_88k,173.0182,2026-04-21,74.5,37.500000,CRITICAL,32.9375,18:04:53,2026-04-21 18:04:53,1,1,0,0,"temperature,humidity",4
9,-Oqk1NeaUaPz_f9OSjNd,173.0182,2026-04-21,74.5,36.666660,CRITICAL,33.8125,18:05:07,2026-04-21 18:05:07,1,1,0,0,"temperature,humidity",4
